# CityPulse — Big Data Urban Pollution Intelligence

**Stack:** Hadoop Streaming MapReduce + Hive SQL + Python charts.

**Important:** The notebook's default data are synthetic demonstration values, not official pollution measurements. Replace them with verified CPCB/OpenAQ data before making real-world claims.

Run the cells in order. Colab is an ephemeral single-node environment. This notebook demonstrates Hadoop Streaming with the local runner; HDFS/Hive can require additional setup and compatible versions.

## 1. Install libraries and prepare workspace

In [ ]:
!pip -q install pandas numpy matplotlib
import os, sys, subprocess, urllib.request, tarfile, shutil, glob
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

WORK = Path('/content/citypulse')
for folder in ['data','scripts','output']:
    (WORK/folder).mkdir(parents=True, exist_ok=True)
print('Workspace:', WORK)
!java -version

## 2. Generate a reproducible dataset

In [ ]:
# Set N_ROWS to 10_000 for a quick test, or 100_000+ for a larger demo.
N_ROWS = 100_000
rng = np.random.default_rng(42)
cities = ['Delhi','Mumbai','Pune','Nagpur','Bengaluru','Hyderabad','Chennai','Kolkata']
start = pd.Timestamp('2024-01-01')
dates = start + pd.to_timedelta(rng.integers(0, 730, N_ROWS), unit='D')
city_values = rng.choice(cities, N_ROWS)
base = {'Delhi':82,'Mumbai':34,'Pune':30,'Nagpur':39,'Bengaluru':25,'Hyderabad':37,'Chennai':29,'Kolkata':58}
no2_base = {'Delhi':34,'Mumbai':24,'Pune':18,'Nagpur':20,'Bengaluru':16,'Hyderabad':22,'Chennai':19,'Kolkata':29}
season = np.where(pd.DatetimeIndex(dates).month.isin([11,12,1]), 1.2, 1.0)
pm25 = np.maximum(2, np.array([base[c] for c in city_values])*season + rng.normal(0,17,N_ROWS))
pm10 = np.maximum(4, pm25*1.55 + rng.normal(0,15,N_ROWS))
no2 = np.maximum(1, np.array([no2_base[c] for c in city_values]) + rng.normal(0,6,N_ROWS))
data = pd.DataFrame({
 'city':city_values, 'date':pd.DatetimeIndex(dates).strftime('%Y-%m-%d'),
 'pm25':pm25.round(2), 'pm10':pm10.round(2), 'no2':no2.round(2),
 'temperature_c':rng.normal(28,5,N_ROWS).round(1),
 'humidity_pct':np.clip(rng.normal(61,15,N_ROWS),10,100).round(1)
})
CSV_PATH = WORK/'data'/'citypulse_air_quality.csv'
data.to_csv(CSV_PATH,index=False)
print('Rows:',len(data),'| size MB:',round(CSV_PATH.stat().st_size/1e6,2))
display(data.head())
display(data.isna().sum().to_frame('missing_values'))

## 3. Install Hadoop 3.3.6

In [ ]:
# Downloads Hadoop from Apache archive. If this download fails, check network access or use a Hadoop VM.
HADOOP_VERSION='3.3.6'
HADOOP_HOME=f'/content/hadoop-{HADOOP_VERSION}'
tar_path=f'/content/hadoop-{HADOOP_VERSION}.tar.gz'
if not os.path.isdir(HADOOP_HOME):
    url=f'https://archive.apache.org/dist/hadoop/common/hadoop-{HADOOP_VERSION}/hadoop-{HADOOP_VERSION}.tar.gz'
    urllib.request.urlretrieve(url,tar_path)
    with tarfile.open(tar_path,'r:gz') as tf:
        tf.extractall('/content')
java_home=subprocess.check_output("dirname $(dirname $(readlink -f $(which java)))",shell=True,text=True).strip()
os.environ['JAVA_HOME']=java_home
os.environ['HADOOP_HOME']=HADOOP_HOME
os.environ['PATH']=f'{HADOOP_HOME}/bin:{HADOOP_HOME}/sbin:'+os.environ['PATH']
print(subprocess.run(['hadoop','version'],capture_output=True,text=True).stdout[:500])

## 4. Write Hadoop Streaming mapper and reducer

In [ ]:
mapper_code = r'''#!/usr/bin/env python3
import csv, sys
from datetime import datetime
for row in csv.DictReader(sys.stdin):
    try:
        city=row['city'].strip().replace('|','_').replace('\t',' ')
        month=datetime.strptime(row['date'].strip(),'%Y-%m-%d').strftime('%Y-%m')
        for field,label in [('pm25','PM25'),('pm10','PM10'),('no2','NO2')]:
            raw=row.get(field,'')
            if raw is None or raw.strip()=='':
                continue
            value=float(raw)
            if value>=0:
                print(f'{city}|{month}|{label}\t{value}')
    except (ValueError,KeyError,TypeError):
        continue
'''
reducer_code = r'''#!/usr/bin/env python3
import sys
current_key=None
total=0.0
count=0
def emit(key,total,count):
    if key is not None and count:
        city,month,pollutant=key.split('|',2)
        print(f'{city}\t{month}\t{pollutant}\t{total/count:.3f}\t{count}')
for line in sys.stdin:
    line=line.rstrip('\n')
    if '\t' not in line:
        continue
    key,raw=line.split('\t',1)
    try: value=float(raw)
    except ValueError: continue
    if key!=current_key:
        emit(current_key,total,count)
        current_key,total,count=key,0.0,0
    total+=value
    count+=1
emit(current_key,total,count)
'''
mapper_path=WORK/'scripts'/'mapper.py'
reducer_path=WORK/'scripts'/'reducer.py'
mapper_path.write_text(mapper_code); reducer_path.write_text(reducer_code)
os.chmod(mapper_path,0o755); os.chmod(reducer_path,0o755)
print('Mapper/reducer ready')

## 5. Run MapReduce with Hadoop Streaming (local runner)

In [ ]:
jars=glob.glob(f'{HADOOP_HOME}/share/hadoop/tools/lib/hadoop-streaming-*.jar')
if not jars:
    raise FileNotFoundError('Hadoop Streaming JAR missing')
STREAMING_JAR=jars[0]
OUT=WORK/'output'/'monthly_pollution'
if OUT.exists(): shutil.rmtree(OUT)
cmd=['hadoop','jar',STREAMING_JAR,
 '-D','mapreduce.framework.name=local',
 '-input',str(CSV_PATH),'-output',str(OUT),
 '-mapper',f'{sys.executable} {mapper_path}',
 '-reducer',f'{sys.executable} {reducer_path}',
 '-file',str(mapper_path),'-file',str(reducer_path)]
res=subprocess.run(cmd,capture_output=True,text=True)
print(res.stdout[-1000:])
print(res.stderr[-2500:])
if res.returncode:
    raise RuntimeError('MapReduce job failed; inspect logs above.')
part_files=sorted(OUT.glob('part-*'))
print('Part files:',[p.name for p in part_files])
print('First output lines:')
print(''.join(p.read_text()[:800] for p in part_files[:1]))

## 6. Read output and validate against Pandas

In [ ]:
cols=['city','month','pollutant','avg_value','record_count']
pieces=[pd.read_csv(p,sep='\t',header=None,names=cols) for p in part_files if p.stat().st_size]
monthly=pd.concat(pieces,ignore_index=True)
monthly['avg_value']=pd.to_numeric(monthly['avg_value'])
monthly['record_count']=pd.to_numeric(monthly['record_count']).astype('int64')
display(monthly.head(10))
raw=data.copy()
raw['month']=raw['date'].str[:7]
check=raw.groupby(['city','month'])['pm25'].agg(pandas_mean='mean',raw_count='count').reset_index()
joined=monthly[monthly.pollutant.eq('PM25')].merge(check,on=['city','month'])
joined['abs_diff']=(joined.avg_value-joined.pandas_mean).abs()
print('Max absolute difference vs Pandas (3-decimal reducer output):',round(joined.abs_diff.max(),5))
print('Aggregate rows:',len(monthly))

## 7. Tables and charts

In [ ]:
city_summary=(monthly[monthly.pollutant.eq('PM25')]
 .groupby('city',as_index=False).avg_value.mean()
 .sort_values('avg_value',ascending=False))
display(city_summary.rename(columns={'avg_value':'mean_monthly_pm25'}))
plt.figure(figsize=(10,5))
plt.bar(city_summary.city,city_summary.avg_value)
plt.title('Average Monthly PM2.5 by City — Synthetic Demo')
plt.xlabel('City'); plt.ylabel('PM2.5 (source units)')
plt.xticks(rotation=35,ha='right'); plt.tight_layout(); plt.show()

pm=monthly[monthly.pollutant.eq('PM25')]
pivot=pm.pivot(index='month',columns='city',values='avg_value').sort_index()
plt.figure(figsize=(12,6))
for c in pivot.columns: plt.plot(pivot.index,pivot[c],marker='.',label=c)
plt.title('Monthly PM2.5 Trends — Synthetic Demo')
plt.xlabel('Month'); plt.ylabel('Average PM2.5 (source units)')
plt.xticks(rotation=45); plt.legend(ncol=2); plt.tight_layout(); plt.show()

pollutant_summary=monthly.groupby(['city','pollutant'],as_index=False).avg_value.mean()
display(pollutant_summary.pivot(index='city',columns='pollutant',values='avg_value').round(2))
pollutant_summary.pivot(index='city',columns='pollutant',values='avg_value').plot(kind='bar',figsize=(12,6))
plt.title('Pollutant Comparison — Synthetic Demo')
plt.xlabel('City'); plt.ylabel('Mean of monthly averages')
plt.xticks(rotation=35,ha='right'); plt.tight_layout(); plt.show()

## 8. Save outputs

In [ ]:
monthly.to_csv(WORK/'output'/'citypulse_monthly_aggregates.csv',index=False)
city_summary.to_csv(WORK/'output'/'citypulse_city_pm25_summary.csv',index=False)
# Single TSV file for loading into a configured Hive environment.
monthly[cols].to_csv(WORK/'output'/'monthly_pollution.tsv',sep='\t',index=False,header=False)
print('Saved:')
for p in sorted((WORK/'output').glob('*')):
    if p.is_file(): print(p.name, p.stat().st_size, 'bytes')

## 9. Hive setup and queries

Hive can be sensitive to Java/Hadoop versions in Colab. To avoid falsely claiming success, run the SQL below only in a working Hive installation. Upload the raw CSV to HDFS path `/citypulse/raw` and the MapReduce TSV output to `/citypulse/output`, or edit table locations to match your paths.

```sql
CREATE DATABASE IF NOT EXISTS citypulse;
USE citypulse;

CREATE EXTERNAL TABLE IF NOT EXISTS monthly_pollution (
  city STRING, month STRING, pollutant STRING,
  avg_value DOUBLE, record_count BIGINT
)
ROW FORMAT DELIMITED FIELDS TERMINATED BY '\t'
STORED AS TEXTFILE
LOCATION '/citypulse/output';

SELECT city, month, ROUND(avg_value, 2) AS avg_pm25, record_count
FROM monthly_pollution
WHERE pollutant='PM25'
ORDER BY avg_value DESC;

SELECT month, city, avg_value
FROM (
  SELECT month, city, avg_value,
         ROW_NUMBER() OVER (PARTITION BY month ORDER BY avg_value DESC) AS rn
  FROM monthly_pollution WHERE pollutant='PM25'
) ranked
WHERE rn=1;

SELECT city, pollutant, ROUND(AVG(avg_value),2) AS mean_monthly_level
FROM monthly_pollution
GROUP BY city, pollutant
ORDER BY city, pollutant;
```

## Troubleshooting
- Hadoop download failure: retry on a runtime with internet access, or run in a Hadoop Linux VM.
- Streaming JAR missing: inspect `$HADOOP_HOME/share/hadoop/tools/lib/`.
- MapReduce failure: read stderr printed in Step 5; confirm input headers and Python path.
- Hive unavailable: use the included `sql/citypulse_queries.sql` in your course's Hive environment.
- Never present synthetic data as real pollution measurements.